# 04 — BTS mobility event plots (hurricanes)

This notebook uses processed BTS Daily Mobility and hurricane data (from NB03) to visualise mobility patterns around hurricane landfalls:
1. Event-window plots (±14 days)
2. Year-over-year overlays for all landfall states
3. Baseline comparison (hurricane-year vs. other-year mean)
4. Per-capita YoY overlays
5. Full-year seasonal overlays with landfall markers

All figures are saved to `output/figures/`.

In [1]:
from hurricane_mobility.config import PROCESSED_DIR, FIGURES_DIR, ensure_dirs
from hurricane_mobility.lookups import BTS_METRICS, BAND_SPECS, BAND_PER1000_COLS, BAND_LABELS
from hurricane_mobility.features import (
    join_population, compute_per_capita_metrics,
)
from hurricane_mobility import plotting

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

ensure_dirs()
sns.set_theme(style='whitegrid')

In [2]:
# Load processed Parquets (single load, no duplication)
bts = pd.read_parquet(PROCESSED_DIR / 'bts_state_daily.parquet')
hur = pd.read_parquet(PROCESSED_DIR / 'hurricanes_2019_2024.parquet')

bts['date'] = pd.to_datetime(bts['date'])
bts['year'] = bts['date'].dt.year
bts['state_code'] = bts['State Postal Code'].astype('string')

# Join population if available
pop_path = PROCESSED_DIR / 'state_population_2019_2024.parquet'
if pop_path.exists():
    pop = pd.read_parquet(pop_path)
    bts = join_population(bts, pop)
    bts = compute_per_capita_metrics(bts)
    print(f'Population join coverage: {1 - bts["population"].isna().mean():.1%}')
else:
    print('[WARN] Population data not found — per-capita metrics unavailable.')
    bts['population'] = np.nan

print(f'BTS shape: {bts.shape}')
print(f'Hurricanes: {len(hur)} events')

Population join coverage: 100.0%
BTS shape: (98073, 38)
Hurricanes: 19 events


In [3]:
# Selected hurricanes for detailed analysis
EVENTS_SELECTED = [
    {'name': 'Dorian (NC, 2019)', 'state_code': 'NC', 'landfall': pd.Timestamp('2019-09-06')},
    {'name': 'Laura (LA, 2020)', 'state_code': 'LA', 'landfall': pd.Timestamp('2020-08-27')},
    {'name': 'Ida (LA, 2021)', 'state_code': 'LA', 'landfall': pd.Timestamp('2021-08-29')},
    {'name': 'Ian (FL, 2022)', 'state_code': 'FL', 'landfall': pd.Timestamp('2022-09-28')},
    {'name': 'Idalia (FL, 2023)', 'state_code': 'FL', 'landfall': pd.Timestamp('2023-08-30')},
]

EVENTS_BTS = [
    {'name': 'Laura (LA, 2020)', 'storm_name': 'Laura', 'state_code': 'LA', 'landfall_date': pd.Timestamp('2020-08-27')},
    {'name': 'Zeta (LA, 2020)', 'storm_name': 'Zeta', 'state_code': 'LA', 'landfall_date': pd.Timestamp('2020-10-28')},
    {'name': 'Ida (LA, 2021)', 'storm_name': 'Ida', 'state_code': 'LA', 'landfall_date': pd.Timestamp('2021-08-29')},
    {'name': 'Ian (FL, 2022)', 'storm_name': 'Ian', 'state_code': 'FL', 'landfall_date': pd.Timestamp('2022-09-28')},
]

YOY_YEARS = [2019, 2020, 2021, 2022, 2023]

In [4]:
# BTS event-window plots (extended ±14 days, shaded)
plotting.plot_bts_event_windows(
    bts, EVENTS_BTS, BTS_METRICS,
    save_path_prefix=str(FIGURES_DIR / 'bts_event_'),
)

/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# YoY overlays for all landfall states (anchor MM-DD)
plotting.plot_bts_yoy_all_states(
    bts, hur, BTS_METRICS, YOY_YEARS,
    save_path_prefix=str(FIGURES_DIR / 'bts_yoy_'),
)

/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hu

In [6]:
# Compare general-day baseline vs hurricane-day for each selected event
if bts['population'].notna().any():
    plotting.plot_baseline_comparison(
        bts, hur, EVENTS_SELECTED,
        yoy_years=YOY_YEARS,
        save_path_prefix=str(FIGURES_DIR / 'bts_baseline_'),
    )
else:
    print('[SKIP] Baseline comparison requires population data.')

/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:758: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:

In [7]:
# Per-capita YoY overlays (2019-2023) for selected hurricanes
PERCAPITA_METRICS = [
    'trips_per_1000', 'stay_home_share', 'not_stay_home_share',
    'short_trips_per_1000', 'medium_trips_per_1000', 'long_trips_per_1000',
]

if bts['population'].notna().any():
    plotting.plot_percapita_yoy_overlays(
        bts, EVENTS_SELECTED, PERCAPITA_METRICS, YOY_YEARS,
        save_path_prefix=str(FIGURES_DIR / 'bts_percapita_'),
    )
else:
    print('[SKIP] Per-capita overlays require population data.')

/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Seasonal YoY overlays (full year) for selected states with landfall markers
SEASON_METRICS = ['trips_per_1000', 'stay_home_share']

if bts['population'].notna().any():
    plotting.plot_seasonal_yoy_overlays(
        bts, hur, EVENTS_SELECTED, SEASON_METRICS, YOY_YEARS,
        save_path_prefix=str(FIGURES_DIR / 'bts_seasonal_'),
    )
else:
    print('[SKIP] Seasonal overlays require population data.')

print('\nAll NB04 figures saved to:', FIGURES_DIR)

Seasonal overlays for state FL


/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Seasonal overlays for state LA


/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Seasonal overlays for state NC

All NB04 figures saved to: /Users/yzc/Desktop/CHIP/hurricane/output/figures


/Users/yzc/Desktop/CHIP/hurricane/src/hurricane_mobility/plotting.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
